# LLM vs ML: classifying agent skills (RQ3)

GPT, Gemini and DeepSeek classify the same skills as the ML models (`normal` / `suspicious` / `malicious`). Their
scores, tokens and cost are compared with the ML models on **exactly the same skills**.

| File | What it does |
|---|---|
| `llm_eval/llm_config.py` | **all settings**: the LLMs (model, API key name, prices), dataset choice, sample size |
| `llm_eval/data.py` | loads the exported skills, stratified sample |
| `llm_eval/prompt.py` | the prompt (identical for every LLM) and reading the answers |
| `llm_eval/providers.py` | one class per API (OpenAI, Gemini, OpenAI-compatible = DeepSeek) |
| `llm_eval/runner.py` | sends the skills, collects the answers, records tokens and cost |
| `llm_eval/evaluate.py` | scores LLMs and ML models on the same skills |

**Before the first run**
1. Run `skill_vulnerability_classifier_with_flags_v2.ipynb` up to Section 6, then **Section 10** (writes `llm_eval/data/`).
2. Copy `.env.example` to `.env` and paste your three API keys (the file is git-ignored).
3. Check the model IDs and prices in `llm_eval/llm_config.py`, then set `PRICES_VERIFIED = True`.

**How the batches work.** GPT and Gemini use the providers' Batch APIs (half price, results within 24 hours).
DeepSeek has no batch API, so its requests are sent in parallel and finish in minutes. Everything is saved to
`results/llm/<run>/<llm>/` as it arrives. Re-running a cell continues where it stopped: answered skills are skipped
and a batch is never submitted twice. You can close this notebook while the batches run.

In [ ]:
%pip install -q -r requirements-llm.txt

## 1. Settings

`DATASET = "sample"` (default) classifies a stratified sample of `SAMPLE_SIZE` skills; `"full"` classifies all of them.
`SAMPLE_FROM = "test"` (default) draws from the ML test split (1,100 skills the ML models never saw). Change these here
or in `llm_config.py`.

In [1]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
import os

from llm_eval import data, evaluate, runner
from llm_eval import llm_config as cfg

cfg.DATASET = "sample"        # "sample" (stratified) or "full"
cfg.SAMPLE_SIZE = 1000
cfg.SAMPLE_FROM = "test"      # "test" (fair comparison with ML) or "all" (5,498 skills; ~80% were ML training data)
LLMS_TO_RUN = list(cfg.LLMS)  # e.g. ["gpt"] to run one LLM only

runner.load_env()
missing = [c["api_key_env"] for n, c in cfg.LLMS.items() if n in LLMS_TO_RUN and not __import__("os").environ.get(c["api_key_env"])]
print("API keys missing:", missing or "none")
pd.DataFrame({n: {"provider": c["provider"], "model": c["model"], **{f"$/1M {k}": v for k, v in c["prices"].items()},
                  "batch discount": c.get("batch_discount", 1.0)} for n, c in cfg.LLMS.items()}).T

API keys missing: none


,provider,model,$/1M input,$/1M cached_input,$/1M output,batch discount
deepseek,openai_compatible,deepseek-flash,0.3,None,1.2,1.0


## 2. The skills to classify

The sample keeps the share of every label **and** of every attack pattern inside each label (`STRATIFY_BY`).
The tables show how close the sample is to the pool it was drawn from (difference in percentage points).
The chosen skill IDs are saved to `selected_skills.csv`, so every LLM gets the same skills.

In [2]:
run = runner.Run()
print(f"Run '{run.name}': {len(run.skills):,} skills, {run.skills['truncated'].sum()} longer than "
      f"{cfg.MAX_INPUT_CHARS:,} characters (cut in the middle)\n")
pool, _ = data.load_export()
if cfg.SAMPLE_FROM == "test":
    pool = pool[pool["split"] == "test"]
display(data.representativeness(pool, run.skills, "judgment"))
rep = data.representativeness(pool, run.skills, "primary_pattern")
print(f"Attack patterns: {len(rep)} groups, largest difference {rep['difference'].abs().max():.2f} percentage points")

Run 'test_sample1000_seed42': 1,000 skills, 35 longer than 120,000 characters (cut in the middle)



,population %,sample %,difference
judgment,,,
malicious,51.64,51.6,-0.04
normal,29.91,29.9,-0.01
suspicious,18.45,18.5,0.05


Attack patterns: 38 groups, largest difference 0.06 percentage points


## 3. Expected cost (no API calls)

Rough estimate: input tokens are about characters / 4, and output is about 80 answer tokens plus a guess for hidden
reasoning. The real token counts and cost are recorded per skill after the run.

In [ ]:
runner.estimate(LLMS_TO_RUN, mode="batch")

## 4. Quick test (optional, about 5 cents)

Sends 5 skills (all three labels) to every LLM straight away (full price), to check the keys, model names and answer
format before the big batch. Saved under its own run folder (`..._first5`), so it does not mix with the real run.

In [ ]:
test_run = runner.Run(limit=5)
for llm in LLMS_TO_RUN:
    runner.run_realtime(llm, test_run)
display(pd.read_csv(test_run.dir / LLMS_TO_RUN[0] / "predictions.csv")[["id", "judgment", "prediction", "confidence", "reason", "error", "cost_usd"]])

## 5. Send the run

GPT and Gemini: submits the batch jobs and returns at once (results within 24 hours). DeepSeek: sends the requests
in parallel and returns when they are done.

In [5]:
for llm in LLMS_TO_RUN:
    runner.submit(llm, run)

deepseek: openai_compatible has no batch API, sending parallel requests instead
deepseek: sending 1000 requests (8 in parallel) ...
  50/1000  (9s)
  100/1000  (17s)
  150/1000  (25s)
  200/1000  (33s)
  250/1000  (46s)
  300/1000  (54s)
  350/1000  (63s)
  400/1000  (71s)
  450/1000  (79s)
  500/1000  (87s)
  550/1000  (95s)
  600/1000  (103s)
  650/1000  (111s)
  700/1000  (120s)
  750/1000  (128s)
  800/1000  (136s)
  850/1000  (144s)
  900/1000  (152s)
  950/1000  (160s)
  1000/1000  (167s)
deepseek: 1000/1000 skills answered, 0 failed or unreadable, cost $2.5793 -> results/llm/test_sample1000_seed42/deepseek/predictions.csv


## 6. Check the batches and download the results

Run these cells any time (also after restarting the kernel: run Sections 1–2 first). `collect` downloads finished
jobs and writes `predictions.csv`. If some answers failed, the last cell re-sends only those.

In [ ]:
for llm in LLMS_TO_RUN:
    runner.status(llm, run)

In [ ]:
for llm in LLMS_TO_RUN:
    runner.collect(llm, run)

In [ ]:
# Optional: re-send only the skills whose answer failed (as a new batch for GPT / Gemini), then run `collect` again
for llm in LLMS_TO_RUN:
    runner.submit(llm, run, retry_failed=True)

## 7. LLMs vs ML models

All models are scored on the same skills, with the same metrics as Section 6 of the ML notebook. A skill whose LLM
answer failed counts as wrong. Saved to `results/llm/<run>/comparison_llm_vs_ml.csv` (plus classification reports and
confusion matrices).

In [ ]:
table, reports, cms = evaluate.compare(run.name)
cols = ["type", "model", "macro F1", "macro precision", "macro recall", "accuracy", "F1 normal", "F1 suspicious",
        "F1 malicious", "failed answers", "cost (USD)", "cost per 1k skills (USD)", "est. cost all 5,498 skills (USD)",
        "input tokens", "output tokens", "reasoning tokens", "via"]
display(table[[c for c in cols if c in table.columns]].round(4))

In [ ]:
FIG_DIR = run.dir / "figures"
FIG_DIR.mkdir(exist_ok=True)
TYPE_COLORS = {"ML": "#2a78d6", "LLM": "#eb6834"}     # same palette as the ML notebook
INK, MUTED, GRID = "#1f1f1f", "#6b6b6b", "#e6e6e6"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight", "font.size": 10,
                     "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True, "xtick.color": MUTED,
                     "ytick.color": MUTED, "text.color": INK, "legend.frameon": False})

# Macro F1 of every model (highest at the top)
t = table.sort_values("macro F1")
kind = t["type"].str.split().str[0]
fig, ax = plt.subplots(figsize=(8, 0.42 * len(t) + 1.2))
ax.barh(t.index, t["macro F1"].astype(float), color=kind.map(TYPE_COLORS), height=0.6)
for y, v in enumerate(t["macro F1"].astype(float)):
    ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=9, color=INK)
ax.set_xlim(0, 1.08); ax.grid(axis="y", visible=False)
ax.set_xlabel("macro F1 (same skills for every model)")
ax.set_title(f"LLMs vs ML models: macro F1 on {int(t['skills scored'].iloc[0])} skills")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in TYPE_COLORS.values()], labels=list(TYPE_COLORS),
          loc="lower right")
fig.savefig(FIG_DIR / "01_macro_f1_llm_vs_ml.png"); plt.show()

# Cost of the LLMs (a separate chart: different unit, never a second y-axis)
llm_t = table[table["type"] == "LLM"].sort_values("cost per 1k skills (USD)")
fig, ax = plt.subplots(figsize=(8, 0.42 * len(llm_t) + 1.2))
ax.barh(llm_t.index, llm_t["cost per 1k skills (USD)"].astype(float), color=TYPE_COLORS["LLM"], height=0.6)
for y, v in enumerate(llm_t["cost per 1k skills (USD)"].astype(float)):
    ax.text(v, y, f"  ${v:.2f}", va="center", fontsize=9, color=INK)
ax.grid(axis="y", visible=False); ax.set_xlabel("US$ per 1,000 skills (as run: batch price where used)")
ax.set_title("LLM cost per 1,000 skills")
ax.set_xlim(0, llm_t["cost per 1k skills (USD)"].astype(float).max() * 1.25)
fig.savefig(FIG_DIR / "02_llm_cost_per_1k.png"); plt.show()

In [ ]:
# Confusion matrices (share of each true class), LLMs first, then the best ML model
BLUES = LinearSegmentedColormap.from_list("blues", ["#f7fbff", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
best_ml = table.index[table["type"] == "ML (best)"].tolist()
show = [n for n in table.index if n.startswith("LLM")] + best_ml
fig, axes = plt.subplots(1, len(show), figsize=(4.6 * len(show), 4.2), squeeze=False)
for ax, name in zip(axes[0], show):
    cm = cms[name].values
    pct = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(pct, cmap=BLUES, vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]}\n{pct[i, j]:.0%}", ha="center", va="center", fontsize=9,
                    color="white" if pct[i, j] > 0.55 else INK)
    ax.set_xticks(range(3)); ax.set_xticklabels(evaluate.LABELS); ax.set_yticks(range(3))
    ax.set_yticklabels(evaluate.LABELS if ax is axes[0][0] else [])   # class names once, on the left
    ax.set_xlabel("predicted"); ax.grid(False); ax.set_title(name)
axes[0][0].set_ylabel("true")
# failed LLM answers ("unparsed") are not a column here, so a row can add up to less than 100%
fig.savefig(FIG_DIR / "03_confusion_llm_vs_best_ml.png"); plt.show()

In [ ]:
for name, rep in reports.items():
    print(f"==================== {name} ====================")
    print(rep)

## 8. What the LLMs got wrong

The skills an LLM misclassified, with its one-sentence reason. Useful for the error analysis in the write-up.

In [ ]:
for llm in LLMS_TO_RUN:
    p = pd.read_csv(run.dir / llm / "predictions.csv")
    wrong = p[p["prediction"] != p["judgment"]]
    print(f"{llm}: {len(wrong)} wrong of {len(p)}")
    display(wrong[["id", "judgment", "prediction", "confidence", "reason", "truncated"]].head(10))